# Open-Coding
Do open-coding using an LLM. Switch the MODEL and reasoning vs CoT

In [ ]:
import json
from tqdm import tqdm
import pandas as pd

from src.datasets import get_or_create_dataset
from src.model_configurations import deepseek_reasoner
from src.prompt_util import prompt_deepseek

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)

### Ask LLM to do open-coding and find common strategies

In [ ]:
def format_examples(df, n: int = 5):
    examples = df.sample(min(len(df), n))
    return [{
        "id": e["Id"],
        "problem": e["problem"],
        "reasoning": e["reasoning"]
    } for i,e in examples.iterrows()]

def discover_categories_from_traces(examples):
    """
    examples: list of dicts like:
        [
            {"id": "1", "problem": "...", "reasoning": "..."},
            {"id": "2", "problem": "...", "reasoning": "..."},
            ...
        ]
    """
    
    # Build a single text block with the examples
    traces_text = []
    for ex in examples:
        traces_text.append(
            f"Trace {ex['id']}:\nProblem: {ex['problem']}\nReasoning:\n{ex['reasoning']}\n"
        )
    traces_text = "\n\n".join(traces_text)

    system_prompt = (
        "You are an expert in cognitive task analysis, think-aloud protocol analysis, "
        "and grounded theory coding. Your task is to derive inductive categories of cognitive "
        "processes from reasoning traces of experts that are generating math distractors for a multiple choice exam. Follow qualitative analysis best practices: "
        "bottom-up coding, constant comparison, memoing, and grounding all claims in the data."
    )

    user_prompt = f"""
I will provide you with a list of reasoning traces. Your task is to discover
common categories of reasoning or cognitive behaviors of the experts that are generating math distractors, using a systematic inductive approach.

Your responsibilities:

1. Identify recurring cognitive behaviors or steps.
2. For every category, provide:
   - A clear definition (1-2 sentences)
   - A description of what behaviors fall under it
3. Provide 2-3 grounded citations for each category:
   - Verbatim excerpts from the traces
   - Include trace ID and step number (or text location)
4. Do not invent anything not present in the traces.
5. Focus only on recurring categories, not one-off behaviors.

OUTPUT FORMAT:

## Discovered Categories

### Category 1: [Name]
**Definition:**
[...]
**Example Citations:**
- "..." (Trace X, Step Y)
- "..." (Trace A, Step B)
- "..." (Trace C, Step D)

### Category 2: [Name]
**Definition:**
[...]

(Continue as needed.)

After listing categories, include:

## Notes on Method & Coverage
Explain how the categories were derived and how representative they are.

---

Here are the reasoning traces:

{traces_text}
"""

    return prompt_deepseek(system_prompt, user_prompt, deepseek_reasoner)

In [ ]:
def annotate(responses, results_df, model: str, mode: str):
    datapoints = []
    for k,response in responses.items():
        dp = eedi_dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")
        datapoints.append((k, dp["Problem"]["Question"], response["raw_reasoning"], result["proportional_match"], result["number_correct"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))
    df = pd.DataFrame(datapoints, columns=["Id", "problem", "reasoning", "proportional_match", "number_correct", "distractors", "solvable", "nr_steps_of_solution"])

    # stratification
    unsolvable_df = df[~df["solvable"]]

    high_match_solvable_df = df[(df["proportional_match"] > 0.5) & (df["solvable"])]
    low_match_solvable_df = df[(df["proportional_match"] < 0.5)  & (df["solvable"])]

    high_match_long_problems_solvable_df = df[(df["proportional_match"] > 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]
    low_match_long_problems_solvable_df = df[(df["proportional_match"] < 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]

    # actually run open-coding
    for i in tqdm(range(5)):
        reasoning, answer = discover_categories_from_traces(format_examples(high_match_solvable_df, 10))
        with open(f"eedi_data/category_induction/{mode}/{model}/high_match_solvable_{i}.md", "w") as f:
            f.write(answer)
    
    for i in tqdm(range(5)):
        reasoning, answer = discover_categories_from_traces(format_examples(low_match_solvable_df, 10))
        with open(f"eedi_data/category_induction/{mode}/{model}/low_match_solvable_{i}.md", "w") as f:
            f.write(answer)

    for i in tqdm(range(5)):
        reasoning, answer = discover_categories_from_traces(format_examples(high_match_long_problems_solvable_df, 10))
        with open(f"eedi_data/category_induction/{mode}/{model}/high_match_long_problems_solvable_{i}.md", "w") as f:
            f.write(answer)

    for i in tqdm(range(5)):
        reasoning, answer = discover_categories_from_traces(format_examples(low_match_long_problems_solvable_df, 10))
        with open(f"eedi_data/category_induction/{mode}/{model}/low_match_long_problems_solvable_{i}.md", "w") as f:
            f.write(answer)

    for i in tqdm(range(5)):
        reasoning, answer = discover_categories_from_traces(format_examples(unsolvable_df, 10))
        with open(f"eedi_data/category_induction/{mode}/{model}/unsolvable{i}.md", "w") as f:
            f.write(answer)


### CoT

In [ ]:
with open("eedi_data/joint_results/deepseek-naive-cot-deepseek-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/deepseek-naive-cot-deepseek-chat_results.csv")

annotate(responses, results_df, "deepseek", "cot")

In [ ]:
with open("eedi_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_results.csv")

annotate(responses, results_df, "glm", "cot")

### Reasoning

In [ ]:
with open("eedi_data/joint_results/deepseek-naive-deepseek-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/deepseek-naive-deepseek-reasoner_results.csv")

annotate(responses, results_df, "deepseek", "reasoning")

In [ ]:
with open("eedi_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_results.csv")

annotate(responses, results_df, "glm", "reasoning")